In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report

# Ensure reproducibility across optimization states
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
%matplotlib inline

print("✅ Training and mitigation environment initialized.")

✅ Training and mitigation environment initialized.


In [2]:
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Ingest our validated Parquet files
train_df = pd.read_parquet(PROCESSED_DIR / "train.parquet")
test_df = pd.read_parquet(PROCESSED_DIR / "test.parquet")

print(f"📖 Train set loaded: {train_df.shape[0]:,} rows")
print(f"📖 Test set loaded: {test_df.shape[0]:,} rows")

📖 Train set loaded: 204,277 rows
📖 Test set loaded: 51,070 rows


In [4]:
# Create our categorical and numerical maps based on the dataset features
# For this dataset, we map categorical attributes to clean dummy indicators
X_train = train_df.drop(columns=['Default'])
y_train = train_df['Default']
X_test = test_df.drop(columns=['Default'])
y_test = test_df['Default']

# Isolate the protected demographic attribute for fairness calculations
# 1 = Young Cohort (Protected), 0 = Mature Cohort (Baseline)
X_train['is_young'] = (X_train['Age'] < 30).astype(int)
X_test['is_young'] = (X_test['Age'] < 30).astype(int)

# Identify categorical features to dummy-encode dynamically
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
if categorical_cols:
    if 'LoanID' in categorical_cols:
        categorical_cols.remove('LoanID')
        X_train = X_train.drop(columns=['LoanID'])
        X_test = X_test.drop(columns=['LoanID'])

    X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
    X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)
    # Align columns to guarantee identical shapes
    X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print(f"📐 Extracted Feature Matrix Shape: {X_train.shape[1]} active training vectors.")

📐 Extracted Feature Matrix Shape: 25 active training vectors.


In [5]:
print("⚖️ Calculating Sample Re-weighting Matrix...")

# Total sample count
N = len(train_df)

# Counts for each target outcome
n_fav = len(train_df[train_df['Default'] == 0])
n_unfav = len(train_df[train_df['Default'] == 1])

# Counts for each demographic cohort
n_protected = len(train_df[train_df['Age'] < 30])
n_baseline = len(train_df[train_df['Age'] >= 30])

# Intersections (Group X Label combinations)
n_prot_fav = len(train_df[(train_df['Age'] < 30) & (train_df['Default'] == 0)])
n_prot_unfav = len(train_df[(train_df['Age'] < 30) & (train_df['Default'] == 1)])
n_base_fav = len(train_df[(train_df['Age'] >= 30) & (train_df['Default'] == 0)])
n_base_unfav = len(train_df[(train_df['Age'] >= 30) & (train_df['Default'] == 1)])

# Calculate demographic fairness weights
# W = (N_group * N_label) / (N * N_group_label)
w_prot_fav = (n_protected * n_fav) / (N * n_prot_fav) if n_prot_fav > 0 else 1.0
w_prot_unfav = (n_protected * n_unfav) / (N * n_prot_unfav) if n_prot_unfav > 0 else 1.0
w_base_fav = (n_baseline * n_fav) / (N * n_base_fav) if n_base_fav > 0 else 1.0
w_base_unfav = (n_baseline * n_unfav) / (N * n_base_unfav) if n_base_unfav > 0 else 1.0

# Map weights back to the training matrix rows
train_weights = np.ones(N)
train_weights[(train_df['Age'] < 30) & (train_df['Default'] == 0)] = w_prot_fav
train_weights[(train_df['Age'] < 30) & (train_df['Default'] == 1)] = w_prot_unfav
train_weights[(train_df['Age'] >= 30) & (train_df['Default'] == 0)] = w_base_fav
train_weights[(train_df['Age'] >= 30) & (train_df['Default'] == 1)] = w_base_unfav

X_train['sample_weight'] = train_weights
print("✅ Mitigation weights generated and mapped to training space.")

⚖️ Calculating Sample Re-weighting Matrix...
✅ Mitigation weights generated and mapped to training space.


In [6]:
# Extract sample weights and remove them from the active training features
weights = X_train['sample_weight'].values
X_train_model = X_train.drop(columns=['sample_weight'])
X_test_model = X_test.copy()

# 1. Train the Baseline Model (completely ignoring sample weights)
print("🌲 Training Baseline Model...")
base_model = XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss', n_estimators=100)
base_model.fit(X_train_model, y_train)

# 2. Train the Fair/Mitigated Model (passing our sample weights into the loss function)
print("⚖️ Training Mitigated (Fair) Model...")
fair_model = XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss', n_estimators=100)
fair_model.fit(X_train_model, y_train, sample_weight=weights)

print("✅ Both models successfully trained.")

🌲 Training Baseline Model...
⚖️ Training Mitigated (Fair) Model...
✅ Both models successfully trained.


In [7]:
def audit_model_fairness(model, X, y, description):
    # Predict probabilities and hard labels
    preds_proba = model.predict_proba(X)[:, 1]
    preds_labels = model.predict(X)
    
    auc = roc_auc_score(y, preds_proba)
    
    # Calculate selection rate (approval rate) for both cohorts
    # In lending, "Approval" means predicting No Default (0)
    df_audit = pd.DataFrame({'is_young': X['is_young'], 'pred_default': preds_labels})
    df_audit['approved'] = (df_audit['pred_default'] == 0).astype(int)
    
    approval_rates = df_audit.groupby('is_young')['approved'].mean()
    
    # Disparate Impact Ratio = (Approval Rate of Protected) / (Approval Rate of Baseline)
    # Target regulatory window is 0.80 - 1.25
    di_ratio = approval_rates[1] / approval_rates[0] if approval_rates[0] > 0 else 0
    
    print(f"=== 📊 Audit Metrics for: {description} ===")
    print(f"   Predictive Power (ROC-AUC): {auc:.4f}")
    print(f"   Young Cohort Approval Rate: {approval_rates[1]*100:.2f}%")
    print(f"   Mature Cohort Approval Rate: {approval_rates[0]*100:.2f}%")
    print(f"   ⚠️ Disparate Impact Ratio:  {di_ratio:.4f}")
    
    if 0.80 <= di_ratio <= 1.25:
        print("   ✅ COMPLIANCE STATUS: Fair Decision Boundaries Achieved.")
    else:
        print("   🚨 COMPLIANCE STATUS: Fails Fairness Audit (Violates 4/5ths Rule).")
    print("\n")

# Run the comparative fairness audit
audit_model_fairness(base_model, X_test_model, y_test, "Baseline Model (Unmitigated)")
audit_model_fairness(fair_model, X_test_model, y_test, "Mitigated Model (Sample Re-weighted)")

=== 📊 Audit Metrics for: Baseline Model (Unmitigated) ===
   Predictive Power (ROC-AUC): 0.7424
   Young Cohort Approval Rate: 95.24%
   Mature Cohort Approval Rate: 99.12%
   ⚠️ Disparate Impact Ratio:  0.9608
   ✅ COMPLIANCE STATUS: Fair Decision Boundaries Achieved.


=== 📊 Audit Metrics for: Mitigated Model (Sample Re-weighted) ===
   Predictive Power (ROC-AUC): 0.7219
   Young Cohort Approval Rate: 98.43%
   Mature Cohort Approval Rate: 98.58%
   ⚠️ Disparate Impact Ratio:  0.9984
   ✅ COMPLIANCE STATUS: Fair Decision Boundaries Achieved.


